In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
noodulz_pokemon_dataset_1000_path = kagglehub.dataset_download('noodulz/pokemon-dataset-1000')

print('Data source import complete.')


100%|██████████| 785M/785M [00:43<00:00, 19.1MB/s]

Extracting files...


Data source import complete.


### Neccessary Imports

In [2]:
import torch.nn as nn
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import transforms
from PIL import Image
import pandas as pd
from pathlib import Path
import os

### Path

In [10]:
import kagglehub
path = kagglehub.dataset_download("noodulz/pokemon-dataset-1000")

Using Colab cache for faster access to the 'pokemon-dataset-1000' dataset.


In [11]:
train_data_path = Path("/kaggle/input/pokemon-dataset-1000/pokemon-dataset-1000/train")
test_data_path = Path("/kaggle/input/pokemon-dataset-1000/pokemon-dataset-1000/test")


In [12]:
img_width = 128
img_height = 128

### Transformation

In [13]:
train_transform = transforms.Compose(transforms=
                                    [
                                        transforms.Resize((img_width, img_height)),
                                        transforms.RandomHorizontalFlip(0.5),
                                        transforms.ColorJitter(
                                            brightness=0.3,
                                            contrast=0.3,
                                            saturation=0.3
                                        ),
                                        transforms.RandomRotation(degrees=30),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
                                    ])

test_transform = transforms.Compose(transforms=
                                    [
                                        transforms.Resize((img_width, img_height)),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
                                    ])


### CustomDataClass

In [14]:
class CustomDataClass(Dataset):
    def __init__(self, dir_path, transforms = None):
        self.dir_path = dir_path
        self.transforms = transforms
        self.classes = sorted(os.listdir(dir_path))
        self.img_path = []
        self.img_labels = []
        self.class_names = sorted([d.name for d in self.dir_path.iterdir() if d.is_dir()])

        for label, class_names in enumerate(self.class_names):
            class_dir = self.dir_path/class_names
            for img_file in class_dir.iterdir():
                if img_file.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.img_path.append(str(img_file))
                    self.img_labels.append(label)
            print(f"✅ Loaded {len(self.img_path)} images from {len(self.class_names)} classes")

    def __len__(self):
        return len(self.img_path)

    def __getitem__(self,index):
        image = Image.open(self.img_path[index]).convert("RGB")
        if self.transforms:
            image = self.transforms(image)
        return image, self.img_labels[index]

### Collate Function

In [15]:
def collate_fn(batch):
    image, label = zip(*batch)
    image = torch.stack(image)
    label = torch.tensor(label)
    return image, label


In [16]:
train_data = CustomDataClass(train_data_path, train_transform)
test_data = CustomDataClass(test_data_path, test_transform)

train_loader = DataLoader(train_data, batch_size=32, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, collate_fn=collate_fn, shuffle=False)

✅ Loaded 32 images from 1000 classes
✅ Loaded 64 images from 1000 classes
✅ Loaded 94 images from 1000 classes
✅ Loaded 110 images from 1000 classes
✅ Loaded 122 images from 1000 classes
✅ Loaded 154 images from 1000 classes
✅ Loaded 184 images from 1000 classes
✅ Loaded 216 images from 1000 classes
✅ Loaded 248 images from 1000 classes
✅ Loaded 258 images from 1000 classes
✅ Loaded 274 images from 1000 classes
✅ Loaded 304 images from 1000 classes
✅ Loaded 316 images from 1000 classes
✅ Loaded 348 images from 1000 classes
✅ Loaded 364 images from 1000 classes
✅ Loaded 396 images from 1000 classes
✅ Loaded 400 images from 1000 classes
✅ Loaded 430 images from 1000 classes
✅ Loaded 440 images from 1000 classes
✅ Loaded 450 images from 1000 classes
✅ Loaded 462 images from 1000 classes
✅ Loaded 494 images from 1000 classes
✅ Loaded 498 images from 1000 classes
✅ Loaded 530 images from 1000 classes
✅ Loaded 555 images from 1000 classes
✅ Loaded 571 images from 1000 classes
✅ Loaded 587 im

### PokeDex Model

In [17]:
class Pokedex_model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.LeakyReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, padding=1),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.LeakyReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, padding=1),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.LeakyReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, padding=1),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.LeakyReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2, padding=1)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )


    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x


In [18]:
num_classes = len(train_data.classes)
device = "cuda" if torch.cuda.is_available() else "cpu"
loss = nn.CrossEntropyLoss()
model = Pokedex_model(num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
model


Pokedex_model(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01)
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): LeakyReLU(negative_slope=0.01)
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): LeakyReLU(negative_slope=0.01)
    (10): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): Leak

In [19]:
Epochs = 10

### Training Function

In [20]:
def train(dataloader, epochs, model):
    model = model
    running_loss = 0
    for epoch in range(epochs):
        model.train()
        for _, (x,y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            yhat = model(x)
            criterion = loss(yhat, y)
            criterion.backward()
            optimizer.step()
            running_loss+=criterion.item()
            print(f"Epoch {epoch}/{epochs} | Loss {criterion.item()}")

    torch.save(model.state_dict(), "pokedex_model.pth")
    print("✅ Model is Saved")
    return f"Total number of Loss is - {round(running_loss,2)}"


In [21]:
train(train_loader, Epochs, model)

Streaming output truncated to the last 5000 lines.
Epoch 2/10 | Loss 4.952435493469238
Epoch 2/10 | Loss 4.68133544921875
Epoch 2/10 | Loss 4.859502792358398
Epoch 2/10 | Loss 4.8181538581848145
Epoch 2/10 | Loss 4.974456787109375
Epoch 2/10 | Loss 4.624457359313965
Epoch 2/10 | Loss 4.8630170822143555
Epoch 2/10 | Loss 4.639180660247803
Epoch 2/10 | Loss 4.521939277648926
Epoch 2/10 | Loss 4.787128448486328
Epoch 2/10 | Loss 4.771339416503906
Epoch 2/10 | Loss 4.491865634918213
Epoch 2/10 | Loss 4.6743364334106445
Epoch 2/10 | Loss 4.663134574890137
Epoch 2/10 | Loss 4.925505638122559
Epoch 2/10 | Loss 4.569893836975098
Epoch 2/10 | Loss 4.927929878234863
Epoch 2/10 | Loss 4.84869384765625
Epoch 2/10 | Loss 4.748235702514648
Epoch 2/10 | Loss 5.028831481933594
Epoch 2/10 | Loss 5.121288299560547
Epoch 2/10 | Loss 4.7081098556518555
Epoch 2/10 | Loss 5.044979095458984
Epoch 2/10 | Loss 5.113069534301758
Epoch 2/10 | Loss 4.761876106262207
Epoch 2/10 | Loss 5.18747615814209
Epoch 2/10 |

'Total number of Loss is - 23396.49'